# M7.4 — Flat catalog compatible with lazy loading

Plan: [`plans/milestone_07/07_catalog_task_dataset_plan.md`](../../plans/milestone_07/07_catalog_task_dataset_plan.md).  
Next: `07_5_task_dataset.ipynb`.

Flatten workbook jobs + optional manifests into `CatalogRow` fields. Image roles become lightweight `RoleRef(manifest_path, role_name)` handles — no arrays at catalog build time.

Roles (M6 contract): `clean`, `particle`, `observed`, optional `anomaly` preview.  
Labels: `n_particles` / ordered `particles`; scalars `particle_x/y/z/radius` only when `n_particles == 1`.


In [1]:
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("Could not locate repository root containing pyproject.toml")
    ROOT = ROOT.parent

!pip install --quiet --no-cache-dir "{ROOT}[dl,dev]" -c "{ROOT}/requirements.txt"

from gummybear.paths import display_path

print(f"ROOT={display_path(ROOT)}")



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
ROOT=.


In [2]:
from tomography_ml_validation.milestone_07 import validation_fixture_paths

paths = validation_fixture_paths()
VALIDATION_ROOT = paths["validation_root"]
WORKBOOK_PATH = paths["workbook_path"]
OUTPUT_ROOT = paths["output_root"]
CACHE_ROOT = paths["cache_root"]
print(f"workbook={display_path(WORKBOOK_PATH)}")


workbook=venv/lib/python3.12/site-packages/tomography_ml_validation/test_data/configs/m6/m6_matrix_plan.xlsx


In [3]:
from IPython.display import display

from tomography_ml.gummybear_data_catalog import (
    build_catalog_rows,
    filter_schedule_consistent,
    load_catalog_jobs,
)
import tomography_ml_validation.milestone_07.validation as m7_validation
from tomography_ml_validation.milestone_07 import (
    assert_single_particle_flat_labels,
    build_demo_multi_particle_catalog_rows,
    catalog_rows_dataframe,
    particle_labels_dataframe,
    write_demo_multi_particle_workbook,
)
from gummybear_validation.notebook_tools import run_installed_pytest_test


## Flat catalog from schedule-consistent subset


In [4]:
catalog_jobs = load_catalog_jobs(WORKBOOK_PATH, VALIDATION_ROOT)
subset_012 = filter_schedule_consistent(catalog_jobs, camera_schedule_id="orbit_matrix_012")
catalog_rows = build_catalog_rows(subset_012)
display(catalog_rows_dataframe(catalog_rows))
assert_single_particle_flat_labels(catalog_rows)
print("flat catalog single-particle label checks passed")


,sample_id,sequence_id,camera_schedule_id,frame_count,n_particles,particle_group_id,particle_radius,diffusion_setup_id,schema_version,composition_domain,field_status
0,0,bear_m6_matrix_002,orbit_matrix_012,12,1,particle_matrix_sphere_001,3.0,diff_matrix_robin_l05,1.3-m6-draft,linear_camera_intensity_before_jpeg,complete
1,1,bear_m6_matrix_003,orbit_matrix_012,12,1,particle_matrix_sphere_001,3.0,diff_matrix_robin_l12,1.3-m6-draft,linear_camera_intensity_before_jpeg,complete


flat catalog single-particle label checks passed


## Multi-particle catalog illustration (planning labels only)

Temporary two-sphere workbook — catalog labels only, no FEM regeneration. Scalars stay `None` when `n_particles > 1`; use `row.particles`.


In [5]:
multi_path = ROOT / "data" / "generated" / "_tmp_m7_multi_particle_catalog.xlsx"
write_demo_multi_particle_workbook(
    multi_path,
    particle_group_id="catalog_two_sphere",
    centers=[(-5.0, 0.5, 2.5), (5.0, -0.5, 2.5)],
    repo_root=ROOT,
)
multi_rows = build_demo_multi_particle_catalog_rows(multi_path, repo_root=ROOT)
row = multi_rows[0]
assert row.n_particles == 2
assert row.particle_group_id == "catalog_two_sphere"
assert row.particle_x is None
display(particle_labels_dataframe(row))
print("multi-particle catalog label checks passed")


,sequence_id,n_particles,particle_group_id,particle_setup_id,center_x,center_y,center_z,radius
0,bear_m6_smoke_001,2,catalog_two_sphere,catalog_two_sphere_p000,-5.0,0.5,2.5,3.0
1,bear_m6_smoke_001,2,catalog_two_sphere,catalog_two_sphere_p001,5.0,-0.5,2.5,3.0


multi-particle catalog label checks passed


## Flat catalog validation


In [6]:
run_installed_pytest_test(
    m7_validation,
    "test_m7_4_flat_catalog_joins_jobs_and_manifests_without_loading_tensors",
)


M7.4
Test executed: test_m7_4_flat_catalog_joins_jobs_and_manifests_without_loading_tensors()

pytest:
../../venv/lib/python3.12/site-packages/tomography_ml_validation/milestone_07/validation.py . [100%]
============================== 1 passed in 1.79s ===============================

Test proves: Catalog construction joins workbook jobs and optional manifests without treating
             samples as tensors.
